# WarisAI — Eksperimen B0, B1, B2 di Kaggle (GPU T4)

Notebook ini menjalankan **B0 (Direct LLM)**, **B1 (Structured Prompt)**, dan **B2 (Naive RAG: Dense + Fixed Chunk)** menggunakan **1 base model yang sama** (`Qwen/Qwen2.5-1.5B-Instruct`) pada split `dev.jsonl`.

### Cara Pakai di Kaggle:
1. Aktifkan **Accelerator → GPU T4 x2** (atau T4/P100) & **Internet ON**.
2. Upload folder/zip repo (`benchmark/dev.jsonl`, `configs/prompts.json`, `warisai_dataset/indexes/fixed/dense.faiss`, `warisai_dataset/indexes/fixed/chunks_meta.jsonl`) sebagai Kaggle Dataset, lalu sesuaikan `DATA_DIR` di sel 2.
3. Jalankan semua sel berurutan. Hasil evaluasi dan raw output tersimpan di `/kaggle/working/runs/`.

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu

In [ ]:
import json, time, re
from pathlib import Path
import torch, faiss, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# Sesuaikan path dataset Kaggle kamu di sini
DATA_DIR = Path('/kaggle/input/datasets/kayistar/dataset-nlp')
OUT_DIR = Path('/kaggle/working/runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # Base generator tetap sama untuk B0-B5
EMBED_MODEL = 'intfloat/multilingual-e5-small'
SEED = 42
TOP_K = 4

torch.manual_seed(SEED)
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
def find_file(root: Path, name: str) -> Path:
    hits = list(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f'{name} tidak ditemukan di {root}')
    return hits[0]

prompts = json.loads(find_file(DATA_DIR, 'prompts.json').read_text(encoding='utf-8'))
dev_queries = [json.loads(l) for l in find_file(DATA_DIR, 'dev.jsonl').open(encoding='utf-8')]
chunks = [json.loads(l) for l in find_file(DATA_DIR, 'chunks_meta.jsonl').open(encoding='utf-8')]
faiss_idx = faiss.read_index(str(find_file(DATA_DIR, 'dense.faiss')))
embedder = SentenceTransformer(EMBED_MODEL)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model.eval()
print(f'Loaded {len(dev_queries)} dev queries and {len(chunks)} fixed chunks.')

In [ ]:
def generate_reply(system_msg: str, user_msg: str, max_new_tokens: int = 320):
    messages = [{'role': 'system', 'content': system_msg}, {'role': 'user', 'content': user_msg}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    lat = time.perf_counter() - t0
    gen_ids = out[0][inputs.input_ids.shape[1]:]
    reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return reply, len(inputs.input_ids[0]), len(gen_ids), round(lat, 3)

def retrieve_dense_topk(query: str, k: int = TOP_K):
    q_emb = np.asarray(embedder.encode(['query: ' + query], normalize_embeddings=True), dtype='float32')
    scores, ids = faiss_idx.search(q_emb, k)
    hits = []
    for sc, ci in zip(scores[0], ids[0]):
        if ci >= 0:
            c = chunks[int(ci)]
            hits.append({'chunk_id': c['chunk_id'], 'doc_id': c['doc_id'], 'page': c['page_start'], 'score': float(sc), 'text': c['text']})
    return hits

def try_parse_json(text: str):
    m = re.search(r'\{.*\}', text, flags=re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

In [ ]:
def run_config(cfg_name: str):
    rows = []
    for q in dev_queries:
        retrieved = []
        if cfg_name == 'B0':
            p = prompts['B0_direct']
            user_msg = p['user_template'].format(query=q['query'])
        elif cfg_name == 'B1':
            p = prompts['B1_structured']
            user_msg = p['user_template'].format(query=q['query'])
        else:
            p = prompts['B2_naive_rag']
            retrieved = retrieve_dense_topk(q['query'], TOP_K)
            ctx = '\n\n'.join(f"[{h['doc_id']} | hal {h['page']}] {h['text'][:900]}" for h in retrieved)
            user_msg = p['user_template'].format(context=ctx, query=q['query'])
        raw, in_tok, out_tok, lat = generate_reply(p['system'], user_msg)
        parsed = try_parse_json(raw)
        rows.append({
            'config': cfg_name, 'query_id': q['query_id'], 'category': q['category'],
            'answerability': q['answerability'], 'query': q['query'],
            'retrieved_doc_ids': [h['doc_id'] for h in retrieved],
            'raw_output': raw, 'parsed_json': parsed,
            'in_tokens': in_tok, 'out_tokens': out_tok, 'latency_s': lat
        })
    out_file = OUT_DIR / f'{cfg_name}_dev_outputs.jsonl'
    out_file.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows), encoding='utf-8')

    n = len(rows)
    json_ok = sum(1 for r in rows if r['parsed_json'] is not None) / n
    unans = [r for r in rows if r['answerability'] == 'unanswerable']
    ans = [r for r in rows if r['answerability'] == 'answerable']
    abstain_acc = sum(1 for r in unans if (r['parsed_json'] or {}).get('verdict') == 'unknown') / max(1, len(unans))
    false_refusal = sum(1 for r in ans if (r['parsed_json'] or {}).get('verdict') == 'unknown') / max(1, len(ans))
    has_cit = sum(1 for r in ans if len((r['parsed_json'] or {}).get('citations', [])) > 0) / max(1, len(ans))
    avg_lat = sum(r['latency_s'] for r in rows) / n
    summary = {'config': cfg_name, 'model': MODEL_NAME, 'seed': SEED, 'json_valid': round(json_ok, 3), 'abstain_acc_unans': round(abstain_acc, 3), 'false_refusal_ans': round(false_refusal, 3), 'citation_rate_ans': round(has_cit, 3), 'avg_latency_s': round(avg_lat, 2)}
    print(summary)
    return summary

summaries = [run_config('B0'), run_config('B1'), run_config('B2')]
(OUT_DIR / 'b0_b2_summary.json').write_text(json.dumps(summaries, indent=2), encoding='utf-8')